<a href="https://colab.research.google.com/github/sawicka-byte/GWB43-groundwater-drought-DIPI/blob/main/02_event_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 Event metrics from SGI time series

This notebook derives drought-event metrics from monthly SGI time series.

The workflow:
- loads processed SGI series,
- reshapes SGI-12 columns from wide to long format,
- identifies drought events using the threshold SGI < -1,
- calculates well-level event metrics,
- aggregates the results to cluster level,
- exports the final cluster summary table.

## Inputs and outputs

### Input files
- `data_processed/SGI_monthly.csv`
- `data_input/metadata_piezometers.csv`

### Output file
- `data_processed/drought_metrics_cluster_summary.csv`

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

In [9]:
REPO_NAME = "GWB43-groundwater-drought-DIPI"
REPO_URL = "https://github.com/sawicka-byte/GWB43-groundwater-drought-DIPI.git"

repo_path = Path(f"/content/{REPO_NAME}")

if not repo_path.exists():
    !git clone {REPO_URL}

repo_root = repo_path
print("Repository root:", repo_root)

Repository root: /content/GWB43-groundwater-drought-DIPI


In [10]:
sgi_file = repo_root / "data_processed" / "SGI_monthly.csv"
meta_file = repo_root / "data_input" / "metadata_piezometers.csv"
output_file = repo_root / "data_processed" / "drought_metrics_cluster_summary.csv"

print("SGI file:", sgi_file)
print("Metadata file:", meta_file)
print("Output file:", output_file)

SGI file: /content/GWB43-groundwater-drought-DIPI/data_processed/SGI_monthly.csv
Metadata file: /content/GWB43-groundwater-drought-DIPI/data_input/metadata_piezometers.csv
Output file: /content/GWB43-groundwater-drought-DIPI/data_processed/drought_metrics_cluster_summary.csv


In [11]:
sgi = pd.read_csv(sgi_file)
meta = pd.read_csv(meta_file)

sgi["date"] = pd.to_datetime(sgi["date"])

print("SGI shape:", sgi.shape)
print("Metadata shape:", meta.shape)

SGI shape: (464, 49)
Metadata shape: (16, 10)


In [12]:
display(sgi.head())
display(meta.head())

,date,SGI3_II/1271/1,SGI3_II/1272/1,SGI3_II/1273/1,SGI3_II/1274/1,SGI3_II/1274/2,SGI3_II/1276/1,SGI3_II/1285/1,SGI3_II/906/1,SGI3_II/908/1,...,SGI12_II/1285/1,SGI12_II/906/1,SGI12_II/908/1,SGI12_II/908/2,SGI12_II/521/1,SGI12_II/527/1,SGI12_II/1272/2,SGI12_II/1275/1,SGI12_II/797/1,SGI12_II/1065/1
0,1985-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1985-11-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1985-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1986-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,cluster_id,aquifer_group,piezometer_id,x_coord_2180,y_coord_2180,top_depth_m,bottom_depth_m,well_depth_m,stratigraphy,lithology
0,1,shallow_unconfined,II/1271/1,441727.379465,523964.377980,4.05,12.1,28.0,Q,sand
1,1,shallow_unconfined,II/1272/1,406124.320712,559613.681590,3.00,4.6,5.5,Q,sand
2,1,shallow_unconfined,II/1273/1,457116.257396,519137.508988,1.86,19.0,19.0,Q,sand
3,1,shallow_unconfined,II/1274/1,437254.525319,574337.265338,4.36,23.0,23.0,Q,sand
4,1,shallow_unconfined,II/1274/2,437254.525319,574337.265338,4.36,23.0,23.0,Q,sand


## Selection of SGI-12 series

The input SGI table is stored in wide format. For event-based drought metrics, only the SGI-12 series are used here.

In [13]:
sgi12_cols = [col for col in sgi.columns if col.startswith("SGI12_")]

print("Number of SGI-12 columns:", len(sgi12_cols))
print("Example SGI-12 columns:", sgi12_cols[:5])

sgi12 = sgi[["date"] + sgi12_cols].copy()
display(sgi12.head())

Number of SGI-12 columns: 16
Example SGI-12 columns: ['SGI12_II/1271/1', 'SGI12_II/1272/1', 'SGI12_II/1273/1', 'SGI12_II/1274/1', 'SGI12_II/1274/2']


,date,SGI12_II/1271/1,SGI12_II/1272/1,SGI12_II/1273/1,SGI12_II/1274/1,SGI12_II/1274/2,SGI12_II/1276/1,SGI12_II/1285/1,SGI12_II/906/1,SGI12_II/908/1,SGI12_II/908/2,SGI12_II/521/1,SGI12_II/527/1,SGI12_II/1272/2,SGI12_II/1275/1,SGI12_II/797/1,SGI12_II/1065/1
0,1985-09-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1985-11-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1985-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1986-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Reshape SGI-12 to long format

Each SGI-12 column corresponds to one piezometer. The table is reshaped to long format so that one row corresponds to one piezometer and one month.

In [14]:
sgi12_long = sgi12.melt(
    id_vars="date",
    var_name="series_name",
    value_name="SGI_12"
)

sgi12_long["piezometer_id"] = sgi12_long["series_name"].str.replace("SGI12_", "", regex=False)

sgi12_long = sgi12_long.dropna(subset=["SGI_12"]).copy()
sgi12_long = sgi12_long.sort_values(["piezometer_id", "date"]).reset_index(drop=True)

print("Long SGI-12 shape:", sgi12_long.shape)
display(sgi12_long.head())

Long SGI-12 shape: (3724, 4)


,date,series_name,SGI_12,piezometer_id
0,1996-03-01,SGI12_II/1065/1,0.703923,II/1065/1
1,1996-04-01,SGI12_II/1065/1,0.409806,II/1065/1
2,1996-05-01,SGI12_II/1065/1,0.375204,II/1065/1
3,1996-06-01,SGI12_II/1065/1,0.340602,II/1065/1
4,1996-07-01,SGI12_II/1065/1,0.333394,II/1065/1


## Event metrics at well level

Drought events are defined using the threshold SGI < -1. For each piezometer, the notebook calculates:
- number of events,
- mean event duration,
- maximum event duration,
- mean event severity,
- vulnerability as maximum cumulative event severity.

In [15]:
THRESHOLD = -1.0

def compute_event_metrics(group, value_col="SGI_12", threshold=THRESHOLD):
    x = group[["date", value_col]].dropna().copy()
    x["is_drought"] = x[value_col] < threshold

    if x["is_drought"].sum() == 0:
        return pd.Series({
            "n_events": 0,
            "mean_duration": np.nan,
            "max_duration": np.nan,
            "mean_severity": np.nan,
            "vulnerability_vul": np.nan,
            "min_intensity": np.nan,
            "max_intensity": np.nan,
            "mean_intensity": np.nan
        })

    x["event_id"] = (x["is_drought"] != x["is_drought"].shift()).cumsum()

    drought_only = x[x["is_drought"]].copy()

    durations = drought_only.groupby("event_id").size()
    severities = drought_only.groupby("event_id")[value_col].apply(lambda z: np.abs(z).sum())

    return pd.Series({
        "n_events": len(durations),
        "mean_duration": durations.mean(),
        "max_duration": durations.max(),
        "mean_severity": severities.mean(),
        "vulnerability_vul": severities.max(),
        "min_intensity": drought_only[value_col].min(),
        "max_intensity": drought_only[value_col].max(),
        "mean_intensity": drought_only[value_col].mean()
    })

In [16]:
well_metrics = (
    sgi12_long.groupby("piezometer_id")
    .apply(compute_event_metrics)
    .reset_index()
)

print("Well-level metrics shape:", well_metrics.shape)
display(well_metrics.head())

Well-level metrics shape: (16, 9)


/tmp/ipykernel_13790/3740077729.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_event_metrics)


,piezometer_id,n_events,mean_duration,max_duration,mean_severity,vulnerability_vul,min_intensity,max_intensity,mean_intensity
0,II/1065/1,2.0,23.0,32.0,34.252187,48.949091,-2.121902,-1.001664,-1.489226
1,II/1271/1,1.0,14.0,14.0,47.989536,47.989536,-4.060390,-1.152199,-3.427824
2,II/1272/1,4.0,7.5,11.0,9.874782,16.677152,-1.816564,-1.030497,-1.316638
3,II/1272/2,2.0,14.0,19.0,19.321010,28.391740,-1.782578,-1.014405,-1.380072
4,II/1273/1,1.0,12.0,12.0,45.718843,45.718843,-3.895325,-3.730064,-3.809904


## Join metadata and aggregate to cluster level

In [17]:
well_metrics = well_metrics.merge(
    meta[["piezometer_id", "cluster_id", "aquifer_group"]],
    on="piezometer_id",
    how="left"
)

display(well_metrics.head())

,piezometer_id,n_events,mean_duration,max_duration,mean_severity,vulnerability_vul,min_intensity,max_intensity,mean_intensity,cluster_id,aquifer_group
0,II/1065/1,2.0,23.0,32.0,34.252187,48.949091,-2.121902,-1.001664,-1.489226,3,deep_confined
1,II/1271/1,1.0,14.0,14.0,47.989536,47.989536,-4.060390,-1.152199,-3.427824,1,shallow_unconfined
2,II/1272/1,4.0,7.5,11.0,9.874782,16.677152,-1.816564,-1.030497,-1.316638,1,shallow_unconfined
3,II/1272/2,2.0,14.0,19.0,19.321010,28.391740,-1.782578,-1.014405,-1.380072,2,intermediate_confined
4,II/1273/1,1.0,12.0,12.0,45.718843,45.718843,-3.895325,-3.730064,-3.809904,1,shallow_unconfined


In [18]:
cluster_metrics = (
    well_metrics.groupby(["cluster_id", "aquifer_group"], as_index=False)
    .agg({
        "min_intensity": "mean",
        "max_intensity": "mean",
        "mean_intensity": "mean",
        "n_events": "mean",
        "mean_duration": "mean",
        "max_duration": "mean",
        "mean_severity": "mean",
        "vulnerability_vul": "mean"
    })
)

cluster_metrics = cluster_metrics.sort_values("cluster_id").reset_index(drop=True)

print("Cluster-level metrics shape:", cluster_metrics.shape)
display(cluster_metrics)

Cluster-level metrics shape: (3, 10)


,cluster_id,aquifer_group,min_intensity,max_intensity,mean_intensity,n_events,mean_duration,max_duration,mean_severity,vulnerability_vul
0,1,shallow_unconfined,-2.848290,-1.359588,-2.208779,2.20,12.813333,15.20,29.408475,33.674219
1,2,intermediate_confined,-2.193486,-1.299265,-1.726009,3.75,15.687500,19.25,25.867541,34.028541
2,3,deep_confined,-2.257949,-1.011391,-1.562982,2.00,30.000000,44.00,47.405741,74.909458


## Export cluster summary

In [19]:
cluster_metrics.to_csv(output_file, index=False)

print("Saved:", output_file)

Saved: /content/GWB43-groundwater-drought-DIPI/data_processed/drought_metrics_cluster_summary.csv


## Reproducibility note

This notebook generates the cluster-level drought-event summary used as an intermediate product for manuscript tables, interpretation, and subsequent DIPI-related analyses.